# Smart Semsar — furniture demo on Colab (Gradio)

تجربة كل خطوات الفرش على إعلان واحد: المخطط → العفش → المراجعة → جولة 3D، وكمان قطعة فرش من صورتك (TRELLIS.2).

**What it does:** opens the Gradio demo (`app.py`) with a public link: pick a listing, set your wishes,
upload a photo of your own furniture, and walk through the furnished 3D plan.

| In the app | Needs |
|---|---|
| listing furnished by the rules, with your wishes | nothing |
| **your own piece from a photo**, made 3D by TRELLIS.2 and placed in the room | `HF_TOKEN` |
| your own piece from a `.glb` file | nothing |
| furniture chosen and arranged by AI ("Who arranges the furniture") | `GEMINI_API_KEY` and/or `GROQ_API_KEY` |

The free Hugging Face account gives a few TRELLIS.2 runs per day; when they are used up the app says so.

**Photo → 3D, two ways**
- **TRELLIS.2** on Hugging Face (textured, best looking). Runs on Hugging Face's GPU, so the CPU runtime is enough,
  but a free account only gets about 5 min of GPU a day (the Space reserves 120 s per model).
- **TripoSR on this notebook's GPU** (step 5, no quota at all). Choose `Runtime → Change runtime type → T4 GPU`
  before running. Simpler models, about 20 s each.

**Before you run** — open the key icon (Secrets) on the left, add these with *Notebook access* on:
- `HF_TOKEN` — a Hugging Face **Read** token (https://huggingface.co/settings/tokens) — for run C
- `GEMINI_API_KEY` — optional, free at https://aistudio.google.com/apikey — reads the photo, AI planner
- `GROQ_API_KEY` — optional, free at https://console.groq.com/keys — takes over when Gemini is busy
- `GITHUB_TOKEN` — only if the repository is private (a token with read access to it)

Then `Runtime → Run all`.

## 1. Settings

In [ ]:
REPO = "ghadeersalah42/SmartSemsar"
BRANCH = "furniture-merge"      # the branch with the merged furniture work
PROPERTY_ID = "PROP_1002"       # any property_id from data/final_merged_dataset.csv

## 2. Get the code
Clones the branch from GitHub. If the branch is not on GitHub yet, upload `furniture-merge.bundle`
when asked (it is applied on top of `3D-Plan`).

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:          # secret missing or notebook access off
        return ""

gh = secret("GITHUB_TOKEN")
url = f"https://{gh}@github.com/{REPO}.git" if gh else f"https://github.com/{REPO}.git"
os.chdir("/content")
if os.path.isdir("SmartSemsar"):        # an older copy in this runtime: bring it up to date
    got = subprocess.run(["git", "-C", "SmartSemsar", "fetch", "-q", "origin", BRANCH]).returncode == 0
    if got:
        subprocess.run(["git", "-C", "SmartSemsar", "reset", "-q", "--hard", "FETCH_HEAD"], check=True)
    else:
        print("Could not update the existing copy from GitHub; if you changed the bundle, "
              "use Runtime > Disconnect and delete runtime, then Run all.")
else:
    ok = subprocess.run(["git", "clone", "-q", "--depth", "50", "-b", BRANCH, url, "SmartSemsar"]).returncode == 0
    if not ok:
        print(f"Branch {BRANCH} not found on GitHub: cloning 3D-Plan and applying the bundle instead.")
        subprocess.run(["git", "clone", "-q", "-b", "3D-Plan", url, "SmartSemsar"], check=True)
        from google.colab import files
        print("Upload furniture-merge.bundle:")
        name = next(iter(files.upload()))
        subprocess.run(["git", "-C", "SmartSemsar", "fetch", "-q", f"/content/{name}", f"{BRANCH}:{BRANCH}"], check=True)
        subprocess.run(["git", "-C", "SmartSemsar", "checkout", "-q", BRANCH], check=True)
os.chdir("/content/SmartSemsar")
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 3. Install (about a minute)

In [ ]:
!pip install -q shapely trimesh scipy mapbox_earcut pandas pydantic pillow requests gradio gradio_client

## 4. Keys
Read from Colab Secrets into the environment; only whether each one is set is printed.

In [ ]:
for name in ("HF_TOKEN", "GEMINI_API_KEY", "GROQ_API_KEY"):
    pasted = secret(name)
    value = "".join(pasted.split())          # keys never contain spaces or line breaks
    if value:
        os.environ[name] = value
    note = " (had spaces or line breaks from pasting; cleaned)" if value != pasted.strip() else ""
    print(f"{name:15s} {'set' if value else 'not set'}{note}")

## 5. Optional: TripoSR on this notebook's GPU (no Hugging Face quota)
Only runs on a GPU runtime (`Runtime → Change runtime type → T4 GPU`); on CPU it is skipped.
Takes 3–5 minutes the first time (TripoSR is installed and loaded). Then pick
**"TripoSR on a Colab GPU"** under *Photo to 3D with* in the app.

In [ ]:
import secrets, sys, subprocess, threading, time, os
import torch

if not torch.cuda.is_available():
    print("No GPU in this runtime: TripoSR skipped (TRELLIS.2 and .glb uploads still work).")
else:
    DEVICE = "cuda:0"
    if not os.path.isdir("/content/TripoSR"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/VAST-AI-Research/TripoSR", "/content/TripoSR"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   cwd="/content/TripoSR", check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart",
                    "onnxruntime"], check=True)
    sys.path.insert(0, "/content/TripoSR")

    # --- same model and photo -> GLB code as cv_service/colab_server.ipynb ---
    import io
    import numpy as np, rembg, trimesh
    from PIL import Image
    from tsr.system import TSR
    from tsr.utils import remove_background, resize_foreground

    model = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
    model.renderer.set_chunk_size(8192)
    model.to(DEVICE)
    rembg_session = rembg.new_session()

    def photo_to_glb(image_bytes, mc_resolution=192, foreground_ratio=0.85, strip_background=True):
        image = Image.open(io.BytesIO(image_bytes))
        image.thumbnail((1024, 1024))
        image = remove_background(image.convert("RGB"), rembg_session) if strip_background else image.convert("RGBA")
        image = resize_foreground(image, foreground_ratio)
        rgba = np.array(image).astype(np.float32) / 255.0
        rgb = rgba[:, :, :3] * rgba[:, :, 3:4] + (1 - rgba[:, :, 3:4]) * 0.5
        image = Image.fromarray((rgb * 255.0).astype(np.uint8))
        with torch.no_grad():
            scene_codes = model([image], device=DEVICE)
        try:
            mesh = model.extract_mesh(scene_codes, True, resolution=mc_resolution)[0]
        except TypeError:
            mesh = model.extract_mesh(scene_codes, resolution=mc_resolution)[0]
        mesh.apply_transform(trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0]))
        mesh.apply_transform(trimesh.transformations.rotation_matrix(np.pi / 2, [0, 1, 0]))
        return mesh.export(file_type="glb")

    # --- the same small server, but only on this machine (no ngrok) ---
    import uvicorn
    from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
    from fastapi.responses import Response

    KEY = secrets.token_hex(16)
    server = FastAPI()
    gpu_lock = threading.Lock()

    @server.get("/health")
    def health(x_api_key: str = Header(default="")):
        if x_api_key != KEY:
            raise HTTPException(401, "Wrong or missing X-API-Key.")
        return {"status": "ok", "model": "TripoSR", "gpu": torch.cuda.get_device_name(0)}

    @server.post("/generate")
    def generate(image: UploadFile = File(...), mc_resolution: int = Form(192),
                 strip_background: bool = Form(True), x_api_key: str = Header(default="")):
        if x_api_key != KEY:
            raise HTTPException(401, "Wrong or missing X-API-Key.")
        try:
            with gpu_lock:
                glb = photo_to_glb(image.file.read(), mc_resolution=mc_resolution, strip_background=strip_background)
        except Exception as e:
            torch.cuda.empty_cache()
            raise HTTPException(500, f"{type(e).__name__}: {e}")
        return Response(content=glb, media_type="model/gltf-binary")

    config = uvicorn.Config(server, host="127.0.0.1", port=8002, log_level="warning")
    threading.Thread(target=uvicorn.Server(config).run, daemon=True).start()
    os.environ["SMARTSEMSAR_COLAB_URL"] = "http://127.0.0.1:8002"
    os.environ["SMARTSEMSAR_COLAB_KEY"] = KEY
    time.sleep(2)
    os.chdir("/content/SmartSemsar")
    sys.path.insert(0, "/content/SmartSemsar")
    from backend.services import colab_service
    print("TripoSR ready on this GPU:", colab_service.health())

## 6. Open the Gradio app
Starts the demo and prints a public `https://….gradio.live` link (valid while this notebook runs).
The app also shows below. Photo → 3D takes about 40 s.

In [ ]:
import os, sys
import gradio as gr
os.chdir("/content/SmartSemsar")
sys.path.insert(0, "/content/SmartSemsar")
for name in [m for m in sys.modules if m.split(".")[0] in ("app", "inference", "backend")]:
    del sys.modules[name]      # re-running this cell always uses the current code and keys
gr.close_all()
import app
print(os.popen("git log --oneline -1").read())
app.demo.queue().launch(share=True, height=1100)

---
## Optional: the same runs from the command line
`inference.py` prints each step; the walkthrough is shown inside the notebook.

### Viewer
Serves the `out/` folder on a local port and shows a walkthrough page inside the notebook.
In the viewer: drag to look around, **Walk** + W A S D to move, or open it full screen with the link.

In [ ]:
import subprocess, time
from google.colab import output

server = subprocess.Popen(["python", "-m", "http.server", "8001", "--directory", "out"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.makedirs("out", exist_ok=True)
time.sleep(1)

def show(property_id=PROPERTY_ID, height=720):
    page = f"/{property_id}/walkthrough.html"
    if not os.path.exists("out" + page):
        print("No walkthrough built for", property_id, "- see the messages above.")
        return
    output.serve_kernel_port_as_iframe(8001, path=page, height=height)

### Run A — the listing furnished by the rules

In [ ]:
!python inference.py $PROPERTY_ID --no-open
show()

### Run B — with your wishes
`--density minimal|normal|full`, `--must-have` / `--exclude` catalog names (sofa, desk, tv, armchair ...),
`--dining-seats 4|6`, `--style-brief` free text.

In [ ]:
!python inference.py $PROPERTY_ID --density full --must-have desk armchair --exclude tv --dining-seats 4 --no-open
show()

### Run C — your own furniture from a photo (TRELLIS.2)
Upload a photo of **one** piece, fully visible (a plain background helps).
Set `KIND` (sofa, armchair, bed, desk, ...) and its real `WIDTH_M`; with `GEMINI_API_KEY` set you can leave
`KIND = ""` and the photo is recognised. Takes about 40 s; uses your Hugging Face GPU quota.

In [ ]:
from google.colab import files

KIND = "armchair"     # "" = let Gemini read it from the photo
WIDTH_M = 0.8         # real width in meters; 0 = the catalog piece's width

uploaded = files.upload()
PHOTO = "/content/" + next(iter(uploaded))
with open(PHOTO, "wb") as f:
    f.write(uploaded[next(iter(uploaded))])
opts = (f"--kind {KIND} " if KIND else "") + (f"--width {WIDTH_M} " if WIDTH_M else "")
!python inference.py $PROPERTY_ID --photo "$PHOTO" $opts --generator trellis --no-open
show()

### Run D — Gemini chooses and arranges the furniture
The model writes placement instructions ("sofa on wall B"); the code computes the positions and the
validator checks them. Without `GEMINI_API_KEY` this falls back to the rules and says so.

In [ ]:
STYLE_BRIEF = "warm and bright, light wood, a reading corner in the living room"
!python inference.py $PROPERTY_ID --planner llm --style-brief "$STYLE_BRIEF" --no-open
show()

## Download the results
`staging.json`, `walkthrough.html` (opens offline in any browser) and your custom model.

In [ ]:
import shutil
shutil.make_archive(f"/content/{PROPERTY_ID}_results", "zip", f"out/{PROPERTY_ID}")
files.download(f"/content/{PROPERTY_ID}_results.zip")